## **Automatic Data Profiling**

An automatic profile produces in one command what the *Manual Data Profiling* notebook builds step by step: statistics per attribute, distributions, correlations, missing values, duplicated rows and **alerts**.
The tool used here is `fg-data-profiling`. It is fast, but it does not decide anything for you: every alert must be checked and understood.

**Notebooks of this series:** 1. Manual Data Profiling · 2. Automatic Data Profiling · 3. Checking Issues in Dataframes · 4. Graphics & Illustrations · 5. EDA for Time Series

### **00. Import Libs and the DataSet**

In [ ]:
#Install Dependencies
!pip install fg-data-profiling

In [2]:
#Import Data Profile Tool
from data_profiling import ProfileReport

In [3]:
#Import Specific Libs
from scipy import stats
from google.colab import files;
from collections import Counter;
from IPython.display import IFrame, display, HTML;

In [4]:
#Import the Others Libs
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import re

In [ ]:
#Import Additional Libs
import json

In [6]:
#Ignore Error Message
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [7]:
#Import Dataframe from Github
df = pd.read_csv('https://raw.githubusercontent.com/jrafa1607/DataScience-ML-AI/main/-%20Datasets/credit_data.csv')

**Attention!** The attributes are renamed exactly as in the *Manual Data Profiling* notebook, so every notebook of the series uses the same names: `Index`, `income`, `age`, `loan` and `result`.

In [11]:
#Define New Attributes Names
df.columns = ['id', 'income', 'age', 'loan', 'result']
df.head(3)

,id,income,age,loan,result
0,1,66155.925095,59.017015,8106.532131,0
1,2,34415.153966,48.117153,6564.745018,0
2,3,57317.170063,63.108049,8020.953296,0


In [12]:
#Change the Name of the Specific Attribute
df = df.rename(columns={df.columns[0]: "Index"})
df.head(3)

,Index,income,age,loan,result
0,1,66155.925095,59.017015,8106.532131,0
1,2,34415.153966,48.117153,6564.745018,0
2,3,57317.170063,63.108049,8020.953296,0


### **01. Automatic Data Profiling**

In [26]:
#Automatic Data Profiling
profile = ProfileReport(df, minimal=True);
profile.to_file(output_file="output.html");

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 5/5 [00:00<00:00, 41.51it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

**How to read the report:** *Overview* ( size, missing cells, duplicated rows, memory ), *Variables* ( one block per attribute ), *Interactions* and *Correlations*, *Missing values*, *Sample* and **Alerts** ( what the tool suspects is wrong ).
With `minimal=True` the expensive computations ( correlations, interactions, missing-value diagrams ) are turned off, which makes the profile much faster on large dataframes.

In [ ]:
#Display the Report Inside the Notebook
profile.to_notebook_iframe()

In [ ]:
#Download the Report to Your Computer ( Google Colab )
files.download('output.html')

In [ ]:
#Full Report ( minimal=False ) with a Title
# Includes correlations, interactions, missing-value diagrams and duplicated rows
profile_full = ProfileReport(df, title='Credit Data - Full Profiling Report', minimal=False);
profile_full.to_file(output_file='output_full.html');

In [ ]:
#Read the Report Programmatically ( JSON )
# The report is also a data structure: it can be tested, stored and compared instead of only being read
report = json.loads(profile_full.to_json())
table = report['table']

keys = ['n', 'n_var', 'n_cells_missing', 'p_cells_missing', 'n_duplicates', 'p_duplicates']
print(pd.Series({k: table.get(k) for k in keys}, dtype=object), '\n')

print('Alerts raised by the tool:')
for alert in report.get('alerts', []):
    print('-', alert)

In [ ]:
#Cross-Check the Report against Pandas ( Quality Control of the Tool )
# Never trust a number you did not verify at least once
checks = pd.DataFrame({
    'pandas': [len(df), df.shape[1], int(df.isnull().sum().sum()), int(df.duplicated().sum())],
    'report': [table.get('n'), table.get('n_var'), table.get('n_cells_missing'), table.get('n_duplicates')]
}, index=['rows', 'attributes', 'missing_cells', 'duplicated_rows'])
checks['match'] = checks['pandas'] == checks['report']
print(f"Agreement between pandas and the report: {checks['match'].mean() * 100:.0f}%")
checks

In [ ]:
#Statistics per Attribute from the Report
columns = ['type', 'n_missing', 'p_missing', 'n_distinct', 'mean', 'std', 'min', 'max', 'skewness', 'kurtosis']
pd.DataFrame(report['variables']).T.reindex(columns=columns)

In [ ]:
#Compare Two Groups ( Paid vs Default ) in a Single Report
# Not every version of the library offers compare(), so the cell is protected
try:
    paid_profile = ProfileReport(df[df['result'] == 0], title='Paid', minimal=True);
    default_profile = ProfileReport(df[df['result'] == 1], title='Default', minimal=True);
    comparison = paid_profile.compare(default_profile)
    comparison.to_file('output_comparison.html')
    print('Comparison report saved: output_comparison.html')
except Exception as error:
    print('compare() is not available in this version:', error)

In [ ]:
#Option 02 - Other Automatic Profiling Libraries ( Optional: Remove the # to Run )
# !pip install sweetviz skimpy

# import sweetviz as sv
# sv.analyze(df).show_html('sweetviz_report.html')     # Visual report with target analysis

# from skimpy import skim
# skim(df)                                             # Compact summary in the terminal style

In [ ]:
#Lightweight Automatic Profile with Pure Pandas ( Rules that Generate Alerts )
# The same idea as the report alerts, but every rule is visible and can be changed
def auto_alerts(data, target=None, skew_limit=1, corr_limit=0.9, imbalance_limit=3):
    alerts = []
    numeric = data.select_dtypes(include='number')

    for col in data.columns:
        s = data[col]
        if s.isnull().any():
            alerts.append((col, 'Missing values', f'{int(s.isnull().sum())} ( {s.isnull().mean() * 100:.2f}% )'))
        if s.nunique() <= 1:
            alerts.append((col, 'Constant attribute', 'only one distinct value'))
        if pd.api.types.is_integer_dtype(s) and s.is_unique:
            alerts.append((col, 'Identifier-like', 'all values are unique integers'))

    for col in numeric.columns:
        s = numeric[col].dropna()
        if (s < 0).any():
            alerts.append((col, 'Negative values', f'{int((s < 0).sum())} values'))
        if abs(s.skew()) > skew_limit:
            alerts.append((col, 'Skewed distribution', f'skewness = {s.skew():.2f}'))

    corr = numeric.corr().abs()
    for i, a in enumerate(corr.columns):
        for c in corr.columns[i + 1:]:
            if corr.loc[a, c] > corr_limit:
                alerts.append((f'{a} / {c}', 'High correlation', f'|r| = {corr.loc[a, c]:.2f}'))

    if target is not None:
        share = data[target].value_counts(normalize=True)
        if share.max() / share.min() > imbalance_limit:
            alerts.append((target, 'Imbalanced target', f'{share.max() / share.min():.1f} : 1'))

    if data.duplicated().any():
        alerts.append(('dataframe', 'Duplicated rows', f'{int(data.duplicated().sum())} rows'))

    return pd.DataFrame(alerts, columns=['attribute', 'alert', 'detail'])

auto_alerts(df, target='result')

### **Automatic vs Manual Profiling**

| | Automatic | Manual |
|---|---|---|
| Speed | One command | Many commands |
| Coverage | Wide and standardized | Only what you decide to look at |
| Control | Limited to the library options | Total: any rule, any metric |
| Risk | Accepting alerts without understanding them | Forgetting to check something |

**Good practice:** run the automatic report first to discover where to look, then confirm the important findings manually ( as done in the cross-check above ).
The next notebook, *Checking Issues in Dataframes*, turns the alerts into a systematic check of missing values, impossible values, duplicates and outliers.